In [ ]:
# 1. Import libraries
import tensorflow as tf
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
# 2. Load dataset
train = pd.read_csv("dataset/Fashion/fashion-mnist_train.csv")
test = pd.read_csv("dataset/Fashion/fashion-mnist_test.csv")

In [ ]:
# ==================================================
# 3. DATA PREPROCESSING
# ==================================================

# 3.1 Check missing values
print("\nMissing values in training data:", train.isnull().sum().sum())
print("Missing values in testing data:", test.isnull().sum().sum())

# 3.2 Handle missing values
if train.isnull().sum().sum() > 0:
    train = train.fillna(train.median(numeric_only=True))

if test.isnull().sum().sum() > 0:
    test = test.fillna(test.median(numeric_only=True))


# 3.3 Check duplicate rows
print("\nDuplicate rows in training data:", train.duplicated().sum())
print("Duplicate rows in testing data:", test.duplicated().sum())

# Remove duplicates
train = train.drop_duplicates().reset_index(drop=True)
test = test.drop_duplicates().reset_index(drop=True)


# 3.4 Separate features and labels
x_train = train.iloc[:, 1:].values
y_train = train.iloc[:, 0].values

x_test = test.iloc[:, 1:].values
y_test = test.iloc[:, 0].values


# 3.5 Convert data type
x_train = x_train.astype('float32')
x_test = x_test.astype('float32')


# 3.6 Normalize pixel values
x_train = x_train / 255.0
x_test = x_test / 255.0


# 3.7 Display dataset information
print("\nTraining data:", x_train.shape)
print("Training labels:", y_train.shape)
print("Testing data:", x_test.shape)
print("Testing labels:", y_test.shape)

In [ ]:
# ==================================================
# 4. BUILD AND TRAIN 3 MODELS
# ==================================================

histories = []
results = []

names = ['Normal', 'L2', 'Dropout']

for name in names:

    model = tf.keras.Sequential([
        tf.keras.layers.Input(shape=(784,)),

        tf.keras.layers.Dense(
            128,
            activation='relu',
            kernel_regularizer=tf.keras.regularizers.l2(0.001)
            if name == 'L2' else None
        )
    ])

    if name == 'Dropout':
        model.add(tf.keras.layers.Dropout(0.5))

    model.add(tf.keras.layers.Dense(10, activation='softmax'))

    model.compile(
        optimizer='adam',
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )

    history = model.fit(
        x_train,
        y_train,
        epochs=5,
        batch_size=64,
        validation_split=0.1,
        verbose=1
    )

    loss, acc = model.evaluate(x_test, y_test, verbose=0)

    histories.append(history)
    results.append(acc)

    print(name, "Test Accuracy:", acc * 100, "%")

In [ ]:
plt.figure(figsize=(8,5))

for i, h in enumerate(histories):

    plt.plot(
        h.history['accuracy'],
        label=names[i] + ' Training Accuracy'
    )

    plt.plot(
        h.history['val_accuracy'],
        label=names[i] + ' Validation Accuracy'
    )

plt.title("Accuracy Comparison")
plt.xlabel("Epochs")
plt.ylabel("Accuracy")
plt.legend()
plt.grid()
plt.show()

plt.figure(figsize=(8,5))

for i, h in enumerate(histories):

    plt.plot(
        h.history['loss'],
        label=names[i] + ' Training Loss'
    )

    plt.plot(
        h.history['val_loss'],
        label=names[i] + ' Validation Loss'
    )

plt.title("Loss Comparison")
plt.xlabel("Epochs")
plt.ylabel("Loss")
plt.legend()
plt.grid()
plt.show()